In [39]:
# datamodule.setup("fit")

In [1]:
import lightning as L
from hydra import compose, initialize
import awkward as ak
import numpy as np
import os
from omegaconf import DictConfig
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger  # , CometLogger

from lightning.pytorch.callbacks import TQDMProgressBar, ModelCheckpoint

with initialize(version_base=None, config_path="../config", job_name="test_app"):
    cfg = compose(config_name="main")
from mltau.tools.io.preprocessed_ParTau_dataloader import ParTDataModule
# from mltau.models.ParTau_module import ParTauModule
from mltau.models import MultiParTau_module, SingleParTau_module

cfg.training.dataloader.batch_size = 128



In [9]:
cfg.training.model.name = "MultiParTau"

In [10]:
datamodule = ParTDataModule(cfg)
model_name = cfg.training.model.name
if model_name == "MultiParTau":
    model = MultiParTau_module.ParTauModule(cfg=cfg, input_dim=17, num_dm_classes=6)
elif model_name == "SingleParTau":
    model = SingleParTau_module.ParTauModule(
        cfg=cfg, input_dim=17, num_dm_classes=6, task=cfg.training.model.task
    )

base_output_dir = "/home/laurits/tmp/test_training"
if model_name == "SingleParTau":
    output_dir = os.path.join(base_output_dir, cfg.training.model.task)
else:
    output_dir = base_output_dir

models_dir = os.path.join(output_dir, "models")
log_dir = os.path.join(output_dir, "logs")
tb_log_dir = os.path.join(output_dir, "tensorboard")
os.makedirs(models_dir, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)
os.makedirs(tb_log_dir, exist_ok=True)

cfg.output_dir = output_dir
print("output_dir:", output_dir)

callbacks = [
    ModelCheckpoint(
        dirpath=models_dir,
        monitor="val_losses/loss",
        mode="min",
        save_top_k=1,
        save_weights_only=True,
        filename="ParT-model_best",
    ),
]

trainer = L.Trainer(
    max_epochs=2,
    callbacks=callbacks,
    logger=[
        TensorBoardLogger(
            save_dir=tb_log_dir,
            name="ParTau_experiment",
            default_hp_metric=False,
        ),
    ],
    # Performance optimizations
    accelerator="auto",  # Automatically detect GPU/CPU
    # gradient_clip_val=1.0,  # Stability with variable sequence lengths
    # log_every_n_steps=50,  # Reduce logging overhead
    num_sanity_val_steps=0,  # Skip sanity validation for faster startup
    enable_progress_bar=True,  # Keep enabled for monitoring
    # precision="16-mixed",  # Enable mixed precision for faster training
    overfit_batches=500
)

# trainer.fit(model=model, datamodule=datamodule)


OSError: [Errno 30] Read-only file system: '/home/laurits'

In [2]:
import inspect
from mltau.tools.evaluation import inference
print(inference.__file__)
print(inspect.getsource(inference.create_predictions_file)[:1200])


/home/norman/ml-tau/ml-tau-model/mltau/tools/evaluation/inference.py
def create_predictions_file(
    best_model, input_path: str, model_name: str, cfg: DictConfig
):
    # Load your .pt file and build the dataset
    tensors = torch.load(input_path, weights_only=True)
    tensors = load_tensors(input_path)
    dataset = ParticleTransformerDataset(
        tensors,
        batch_size=cfg.training.dataloader.batch_size,
        shuffle=False,
    )
    # Create DataLoader
    dataloader = DataLoader(dataset, batch_size=None)

    # --- Postprocess and save as {sample}_test.parquet ---

    (
        all_gen_jet_p4,
        all_reco_jet_p4,
        all_gen_jet_tau_p4,
        all_gen_jet_tau_decaymode,
        all_gen_jet_tau_charge,
        all_is_tau,
    ) = ([], [], [], [], [], [])
    all_cand_charges = []
    all_cand_p4 = []
    all_post = []

    def _move_to_device(item, device):
        if isinstance(item, torch.Tensor):
            return item.to(device)
        if isinstance(

In [3]:
from mltau.tools.evaluation import inference

In [4]:
cfg.training.model.name = "SingleParTau"

task_dir_map = {
    "is_tau": "isTau",
    "charge": "charge",
    "decay_mode": "DM",
    "kinematics": "kin",
}

single_task_base_output_dir = "/home/norman/ml-tau/test_inference_single"
single_task_models_dir = "/home/norman/0422"
print("single_task_base_output_dir:", single_task_base_output_dir)
print("single_task_models_dir:", single_task_models_dir)

single_task_base_output_dir: /home/norman/ml-tau/test_inference_single
single_task_models_dir: /home/norman/0422


In [5]:
def run_single_task_inference(task, best_ckpt_path=None, model_name="SingleParTau"):
    cfg.training.model.name = model_name
    cfg.training.model.task = task
    if model_name == "SingleParTau":
        cfg.output_dir = os.path.join(single_task_base_output_dir, task_dir_map[task])
        if best_ckpt_path is None:
            best_ckpt_path = os.path.join(
                single_task_models_dir,
                task_dir_map[task],
                "models",
                "ParT-model_best.ckpt",
            )
    else:
        cfg.output_dir = "/home/norman/ml-tau/test_inference_multi"
    print("cfg.output_dir:", cfg.output_dir)
    print("best_ckpt_path:", best_ckpt_path)

    if os.path.exists(best_ckpt_path):
        print(f"\n[INFO] Running inference on test set using {best_ckpt_path}")
        if model_name == "MultiParTau":
            best_model = MultiParTau_module.ParTauModule.load_from_checkpoint(
                best_ckpt_path, cfg=cfg, input_dim=17, num_dm_classes=6
            )
        elif model_name == "SingleParTau":
            best_model = SingleParTau_module.ParTauModule.load_from_checkpoint(
                best_ckpt_path,
                cfg=cfg,
                input_dim=17,
                num_dm_classes=6,
                task=task,
            )
        else:
            raise ValueError(f"Unknown model '{model_name}' for prediction.")

        inference.create_predictions_files(
            best_model=best_model, model_name=model_name, cfg=cfg
        )
    else:
        print(
            f"[WARNING] Best checkpoint not found at {best_ckpt_path}. Skipping inference."
        )

## SingleParTau Task Inference

In [6]:
task = "kinematics"
run_single_task_inference(task)

cfg.output_dir: /home/norman/ml-tau/test_inference_single/kin
best_ckpt_path: /home/norman/0422/kin/models/ParT-model_best.ckpt

[INFO] Running inference on test set using /home/norman/0422/kin/models/ParT-model_best.ckpt
[INFO] Saved predictions to /home/norman/ml-tau/test_inference_single/kin/predictions/z_test.parquet


In [ ]:
task = "is_tau"
run_single_task_inference(task)

cfg.output_dir: /home/norman/ml-tau/test_inference_single/isTau
best_ckpt_path: /home/norman/0422/isTau/models/ParT-model_best.ckpt

[INFO] Running inference on test set using /home/norman/0422/isTau/models/ParT-model_best.ckpt


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Predicting: |                                                                     | 0/? [00:00<?, ?it/s]

In [12]:
task = "charge"
run_single_task_inference(task)

cfg.output_dir: /home/norman/ml-tau/test_inference_single/charge
best_ckpt_path: /home/norman/0422/charge/models/ParT-model_best.ckpt

[INFO] Running inference on test set using /home/norman/0422/charge/models/ParT-model_best.ckpt


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Predicting: |                                                                     | 0/? [00:00<?, ?it/s]

[INFO] Saved predictions to /home/norman/ml-tau/test_inference_single/charge/predictions/z_test.parquet


In [13]:
task = "decay_mode"
run_single_task_inference(task)

cfg.output_dir: /home/norman/ml-tau/test_inference_single/DM
best_ckpt_path: /home/norman/0422/DM/models/ParT-model_best.ckpt

[INFO] Running inference on test set using /home/norman/0422/DM/models/ParT-model_best.ckpt


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Predicting: |                                                                     | 0/? [00:00<?, ?it/s]

[INFO] Saved predictions to /home/norman/ml-tau/test_inference_single/DM/predictions/z_test.parquet


In [7]:
# Kinematics parquet sanity check: in-memory decode vs written parquet
import inspect
import awkward as ak
from mltau.tools.evaluation.inference import decode_kinematic_predictions
from mltau.tools.general import reinitialize_p4
import torch
from torch.utils.data import DataLoader
from mltau.tools.io.preprocessed_ParTau_dataloader import ParticleTransformerDataset
from mltau.tools.io.general import BatchInputs

if "shuffle" not in inspect.signature(ParticleTransformerDataset.__init__).parameters:
    raise RuntimeError(
        "ParticleTransformerDataset in this kernel does not support shuffle=False. "
        "Restart the kernel so the patched dataloader is imported before running this check."
    )

kin_parquet_path = os.path.join(single_task_base_output_dir, "kin", "predictions", "z_test.parquet")
print("kin_parquet_path:", kin_parquet_path)

kin_tensors = inference.load_tensors(os.path.join(cfg.dataset.data_dir, "z_test.pt"))
kin_dataset = ParticleTransformerDataset(
    kin_tensors,
    batch_size=cfg.training.dataloader.batch_size,
    shuffle=False,
)
kin_dataloader = DataLoader(kin_dataset, batch_size=None)
kin_batch = next(iter(kin_dataloader))
kin_inputs = BatchInputs(*kin_batch)

cfg.training.model.name = "SingleParTau"
cfg.training.model.task = "kinematics"
kin_best_ckpt_path = os.path.join(single_task_models_dir, "kin", "models", "ParT-model_best.ckpt")
kin_model = SingleParTau_module.ParTauModule.load_from_checkpoint(
    kin_best_ckpt_path,
    cfg=cfg,
    input_dim=17,
    num_dm_classes=6,
    task="kinematics",
    map_location="cpu",
)
kin_model.eval()

with torch.no_grad():
    kin_predictions, _, _ = kin_model.forward(kin_batch)

mem_pred_p4 = reinitialize_p4(
    decode_kinematic_predictions(kin_predictions["kinematics"], ak.Array(kin_inputs.reco_jet_p4s))
)
parquet = ak.from_parquet(kin_parquet_path)
parquet_pred_p4 = reinitialize_p4(parquet.tau_p4[: len(mem_pred_p4)])

print("first 5 in-memory pt:", ak.to_numpy(mem_pred_p4.pt[:5]))
print("first 5 parquet pt:", ak.to_numpy(parquet_pred_p4.pt[:5]))
print("first 5 in-memory eta:", ak.to_numpy(mem_pred_p4.eta[:5]))
print("first 5 parquet eta:", ak.to_numpy(parquet_pred_p4.eta[:5]))
print("first 5 in-memory phi:", ak.to_numpy(mem_pred_p4.phi[:5]))
print("first 5 parquet phi:", ak.to_numpy(parquet_pred_p4.phi[:5]))
print("max |pt diff| first batch:", np.max(np.abs(ak.to_numpy(mem_pred_p4.pt - parquet_pred_p4.pt))))
print("max |eta diff| first batch:", np.max(np.abs(ak.to_numpy(mem_pred_p4.eta - parquet_pred_p4.eta))))
print("max wrapped |phi diff| first batch:", np.max(np.abs(np.arctan2(np.sin(ak.to_numpy(mem_pred_p4.phi - parquet_pred_p4.phi)), np.cos(ak.to_numpy(mem_pred_p4.phi - parquet_pred_p4.phi))))))

kin_parquet_path: /home/norman/ml-tau/test_inference_single/kin/predictions/z_test.parquet
first 5 in-memory pt: [36.150894   9.329726   5.723275  15.65682    4.0218563]
first 5 parquet pt: [36.150894   9.329726   5.723275  15.65682    4.0218563]
first 5 in-memory eta: [ 0.2910782 -1.3561839  0.8744515 -1.6206546 -1.68371  ]
first 5 parquet eta: [ 0.2910782 -1.3561839  0.8744515 -1.6206546 -1.68371  ]
first 5 in-memory phi: [ 2.4641216   1.2679007   0.14132635 -2.9237149  -1.4059001 ]
first 5 parquet phi: [ 2.4641216   1.2679007   0.14132635 -2.9237149  -1.4059001 ]
max |pt diff| first batch: 0.0
max |eta diff| first batch: 0.0
max wrapped |phi diff| first batch: 0.0


In [8]:
# Raw kinematics target/prediction diagnostics
import torch
from torch.utils.data import DataLoader
from mltau.tools.io.preprocessed_ParTau_dataloader import ParticleTransformerDataset
from mltau.tools.io.general import BatchInputs

kin_input_path = os.path.join(cfg.dataset.data_dir, "z_test.pt")
print("kin_input_path:", kin_input_path)

kin_tensors = inference.load_tensors(kin_input_path)
kin_dataset = ParticleTransformerDataset(
    kin_tensors,
    batch_size=cfg.training.dataloader.batch_size,
)
kin_dataloader = DataLoader(kin_dataset, batch_size=None)
kin_batch = next(iter(kin_dataloader))
kin_inputs = BatchInputs(*kin_batch)

cfg.training.model.name = "SingleParTau"
cfg.training.model.task = "kinematics"
kin_best_ckpt_path = os.path.join(single_task_models_dir, "kin", "models", "ParT-model_best.ckpt")
print("kin_best_ckpt_path:", kin_best_ckpt_path)

kin_model = SingleParTau_module.ParTauModule.load_from_checkpoint(
    kin_best_ckpt_path,
    cfg=cfg,
    input_dim=17,
    num_dm_classes=6,
    task="kinematics",
    map_location="cpu",
)
kin_model.eval()

with torch.no_grad():
    kin_predictions, kin_targets, _ = kin_model.forward(kin_batch)

raw_pred = kin_predictions["kinematics"].detach().cpu().numpy()
raw_target = kin_targets["kinematics"].detach().cpu().numpy()

print("raw_pred shape:", raw_pred.shape)
print("raw_target shape:", raw_target.shape)
print("first 5 raw predictions:")
print(raw_pred[:5])
print("first 5 raw targets:")
print(raw_target[:5])
print("median |pred - target| per component:")
print(np.median(np.abs(raw_pred - raw_target), axis=0))

kin_input_path: /scratch/persistent/laurits/ml-tau/0412_increased_stats/z_test.pt
kin_best_ckpt_path: /home/norman/0422/kin/models/ParT-model_best.ckpt
raw_pred shape: (128, 5)
raw_target shape: (128, 5)
first 5 raw predictions:
[[-9.0109427e-03  3.7278216e-03 -9.9018589e-04  9.9954164e-01
  -4.0121645e-02]
 [-3.7277222e-02  1.7895214e-03 -1.3497137e-03  9.9927843e-01
  -1.4613228e+00]
 [-1.9466491e-02  1.5778150e-03 -1.3636388e-03  9.9880344e-01
  -4.3506879e-01]
 [-4.5502141e-02  9.2944317e-04 -2.4989247e-04  9.9990761e-01
  -1.2150797e+00]
 [-2.5827987e-02  3.6887191e-03 -1.4565252e-03  1.0003155e+00
  -3.5595991e-02]]
first 5 raw targets:
[[ 5.1812185e-03  3.2082707e-02 -1.6107552e-03  9.9999869e-01
  -3.4125172e-02]
 [-1.3966395e-01  6.2348247e-03  3.7469179e-03  9.9999297e-01
  -1.6760375e+00]
 [ 2.6366280e-02  4.1306019e-04  2.1204432e-02  9.9977517e-01
  -3.7268937e-01]
 [-8.2163095e-02 -7.4511766e-03  2.8047938e-02  9.9960655e-01
  -1.2678983e+00]
 [ 8.3303601e-03 -1.5762091e-

In [8]:
# from tensorboard.backend.event_processing import event_accumulator

# log_dir = "/home/laurits/tmp/speedup_test2/tensorboard/ParTau_experiment/version_0/"

# ea = event_accumulator.EventAccumulator(log_dir)
# ea.Reload()

# # List available scalar tags
# print(ea.Tags()["scalars"])

# # Extract a specific scalar
# scalars = ea.Scalars("train_losses/decay_mode_loss")

# for s in scalars:
#     print(s.step, s.value)